## Importación

In [34]:
from pathlib import Path
import sys
import yaml

import pandas as pd
import numpy as np
import pandas as pd
import random
import collections

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

from sklearn.ensemble import RandomForestClassifier

BASE_DIR = Path().resolve().parent
sys.path.append(str(BASE_DIR/'src'))

## Rutas

In [35]:
DATA_FILE = BASE_DIR / 'data' / 'Feature_Selection' / '01_train.csv' 
CONFIG_FILE = BASE_DIR / 'config' / '01_experimento.yaml'

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_FILE: {DATA_FILE}")
print(f"CONFIG_FILE: {CONFIG_FILE}")

BASE_DIR: /home/jair/Proyectos/Loan_Status_Prediction
DATA_FILE: /home/jair/Proyectos/Loan_Status_Prediction/data/Feature_Selection/01_train.csv
CONFIG_FILE: /home/jair/Proyectos/Loan_Status_Prediction/config/01_experimento.yaml


## Configuraciónde hiperparpametros

In [36]:
with open(CONFIG_FILE, 'r') as f:
    config = yaml.safe_load(f)

seed = config['seed']
random.seed(seed)
np.random.seed(seed)

test_size = config['data_split']['test_size']
random_state = config['seed']
variable_objetivo = config['target_variable']

## Obtención datos

In [37]:
train = pd.read_csv(DATA_FILE)
train.head()

,age,education_level,annual_income,monthly_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,installment,grade_subgrade,...,loan_purpose_Debt consolidation,loan_purpose_Education,loan_purpose_Home,loan_purpose_Medical,loan_purpose_Other,loan_purpose_Vacation,loan_to_income,has_delinquency_history,sevetity_score,payment_income
0,28,2.0,10.751751,8.267079,0.271553,773,29588.02,9.79,951.81,1.0,...,0,1,0,0,0,0,29577.268249,1,1,115.132561
1,34,1.0,10.618642,8.134004,0.263133,775,500.00,9.69,16.06,1.0,...,0,0,1,0,0,0,489.381358,1,2,1.974427
2,60,1.0,9.514391,7.030291,0.175633,748,21230.36,10.29,687.94,1.0,...,0,0,0,0,1,0,21220.845609,1,1,97.853696
3,57,1.0,10.081107,7.596658,0.235072,648,6602.02,14.95,228.70,3.0,...,0,0,0,1,0,0,6591.938893,1,2,30.105342
4,23,1.0,10.966093,8.481375,0.079735,594,17108.03,14.48,588.71,4.0,...,0,0,0,0,1,0,17097.063907,1,1,69.412092


In [38]:
print(train.columns)

Index(['age', 'education_level', 'annual_income', 'monthly_income',
       'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate',
       'installment', 'grade_subgrade', 'num_of_open_accounts',
       'total_credit_limit', 'current_balance', 'delinquency_history',
       'public_records', 'num_of_delinquencies', 'loan_paid_back', 'age_group',
       'gender_Female', 'gender_Male', 'gender_Other', 'loan_term_60',
       'marital_status_Divorced', 'marital_status_Married',
       'marital_status_Single', 'marital_status_Widowed',
       'employment_status_Employed', 'employment_status_Retired',
       'employment_status_Self-employed', 'employment_status_Student',
       'employment_status_Unemployed', 'loan_purpose_Business',
       'loan_purpose_Car', 'loan_purpose_Debt consolidation',
       'loan_purpose_Education', 'loan_purpose_Home', 'loan_purpose_Medical',
       'loan_purpose_Other', 'loan_purpose_Vacation', 'loan_to_income',
       'has_delinquency_history', 's

In [39]:
print(len(train.columns.to_list()))

43


## Random Forest

In [40]:
X_train = train.drop(variable_objetivo, axis = 1)
y_train = train[variable_objetivo]

print(f'Dimensiones X: {X_train.shape}')
print(f'Dimensiones y: {y_train.shape}')

Dimensiones X: (16000, 42)
Dimensiones y: (16000,)


In [41]:
rfc = RandomForestClassifier(
    n_estimators= config['random_forest']['n_estimators'],
    max_leaf_nodes= config['random_forest']['n_estimators'],
    n_jobs= config['random_forest']['n_jobs']
)
scores = {}
rfc.fit(X_train, y_train)
for name, score in zip(train.columns, rfc.feature_importances_):
    scores[score] = name

asc = {k: v for k, v in sorted(scores.items(), key=lambda item: item[0])}
print('Score \t Característica')
for key, value in asc.items():
    print(f'{np.round(key, 4)} \t {value}')

Score 	 Característica
0.001 	 loan_purpose_Other
0.0012 	 gender_Male
0.0013 	 marital_status_Single
0.0018 	 loan_to_income
0.0019 	 loan_term_60
0.0019 	 public_records
0.002 	 loan_purpose_Home
0.0021 	 employment_status_Unemployed
0.0021 	 loan_purpose_Education
0.0024 	 loan_purpose_Debt consolidation
0.0024 	 loan_purpose_Medical
0.0026 	 loan_purpose_Business
0.0027 	 gender_Other
0.003 	 age_group
0.003 	 marital_status_Married
0.0031 	 marital_status_Divorced
0.0031 	 gender_Female
0.0033 	 loan_purpose_Car
0.0066 	 loan_paid_back
0.0082 	 education_level
0.0093 	 num_of_delinquencies
0.0094 	 has_delinquency_history
0.0103 	 delinquency_history
0.0141 	 num_of_open_accounts
0.0202 	 age
0.021 	 loan_amount
0.0215 	 loan_purpose_Vacation
0.0222 	 sevetity_score
0.0225 	 installment
0.0231 	 monthly_income
0.0232 	 annual_income
0.024 	 total_credit_limit
0.0247 	 current_balance
0.0318 	 employment_status_Employed
0.0322 	 grade_subgrade
0.0328 	 interest_rate
0.0424 	 employ